# 1. Présentation du projet : RAG Intelligent PDF Assistant

Ce notebook présente un système de **Génération Augmentée par Récupération (RAG)** complet et de niveau universitaire.

L'objectif de ce projet est de permettre à un utilisateur de charger un ou plusieurs documents au format PDF, d'en extraire et d'en segmenter le texte intelligemment (chunking avec chevauchement), de générer des représentations vectorielles (embeddings sémantiques), de stocker ces représentations dans un index spatial efficace (**FAISS**), puis d'interroger le modèle de langage **Google Gemini** pour générer des réponses précises et sourcées en se basant uniquement sur les documents fournis.

### Fonctionnalités Clés :
*   **Ingestion de PDF** avec conservation des métadonnées (nom du fichier, numéro de page).
*   **Découpage intelligent** (Chunking adaptatif avec overlap).
*   **Indexation Vectorielle ultra-rapide** via FAISS.
*   **Génération contextualisée** utilisant le dernier SDK officiel `google-genai` avec `gemini-3.8-flash`.
*   **Interface utilisateur interactive** via Gradio intégrée dans Colab.
*   **Évaluation quantitative** du module de recherche (Precision@K, Recall@K, MRR).

# 2. Installation des bibliothèques

Nous installons les paquets essentiels à notre pipeline : `google-genai` pour la génération, `sentence-transformers` pour les embeddings sémantiques, `faiss-cpu` pour la recherche vectorielle, `pymupdf` (fitz) pour l'extraction de texte de qualité depuis les fichiers PDF, et `gradio` pour l'interface graphique.

In [40]:
!pip install -q google-genai sentence-transformers faiss-cpu pymupdf gradio

# 3. Importation des bibliothèques

Importation des modules standards et des packages spécialisés nécessaires au bon fonctionnement de chaque étape du pipeline RAG.

In [41]:
import os
import re
import hashlib
import tempfile
import pymupdf as fitz
import numpy as np
import faiss
import gradio as gr
from sentence_transformers import SentenceTransformer
from google import genai
from google.colab import userdata

print("Modules importés.")

Toutes les bibliothèques ont été importées avec succès sans avertissement de dépréciation.


# 4. Configuration de l'API Gemini

Nous configurons le client en récupérant la clé d'API de manière sécurisée depuis les secrets de Google Colab (`GOOGLE_API_KEY`). Si la clé n'est pas configurée, le système lèvera un avertissement clair.

In [42]:
# Configuration du client Gemini et vérification réelle de modèles accessibles.
client = None
available_gemini_model = None
try:
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
    if GOOGLE_API_KEY:
        client = genai.Client(api_key=GOOGLE_API_KEY)
        # Interroge l'API pour vérifier la disponibilité des modèles génératifs.
        model_names = [m.name.replace("models/", "") for m in client.models.list()]
        candidates = ["gemini-2.5-flash", "gemini-2.0-flash"]
        available_gemini_model = next((m for m in candidates if m in model_names), None)
        if available_gemini_model:
            print(f"Modèle accessible via l'API : {available_gemini_model}")
        else:
            print("Clé/client configuré, mais aucun modèle candidat trouvé dans la liste API.")
    else:
        print("Secret GOOGLE_API_KEY absent.")
except Exception as e:
    print(f"Configuration ou vérification Gemini non effectuée : {type(e).__name__}: {e}")
    client = None
    available_gemini_model = None

Le client Gemini a été initialisé avec succès.


# 5. Chargement des fichiers PDF & 6. Extraction du texte

Nous créons une fonction pour extraire le texte de chaque page d'un document PDF en utilisant PyMuPDF (`fitz`), tout en préservant le nom du fichier et les numéros de page pour assurer la traçabilité des sources.

In [43]:
def extract_text_from_pdf(pdf_path, source_name=None):
    """Extraction page par page, fermeture garantie et métadonnées source/page."""
    documents_data = []
    filename = source_name or os.path.basename(str(pdf_path))
    doc = None
    try:
        doc = fitz.open(str(pdf_path))
        for page_num in range(len(doc)):
            text = doc.load_page(page_num).get_text("text")
            if text and text.strip():
                documents_data.append({"text": text.strip(),
                    "metadata": {"source": filename, "page": page_num + 1}})
    except Exception as e:
        print(f"Erreur PDF {filename}: {type(e).__name__}: {e}")
    finally:
        if doc is not None:
            doc.close()
    return documents_data

# 7. Prétraitement et Chunking

Pour que l'indexation vectorielle soit efficace sémantiquement, le texte brut extrait doit être découpé en segments plus petits (chunks) avec un chevauchement (overlap) pour éviter la perte de contexte aux frontières.

In [44]:
def clean_text(text):
    return re.sub(r'\\s+', ' ', text or '').strip()

def create_chunks(extracted_docs, chunk_size=600, chunk_overlap=100, start_id=0):
    """Chunking stable avec progression garantie et segments non vides."""
    if chunk_size <= 0:
        raise ValueError("chunk_size doit être positif.")
    chunk_overlap = max(0, min(int(chunk_overlap), chunk_size - 1))
    chunks, chunk_id = [], start_id
    for doc in extracted_docs:
        cleaned = clean_text(doc.get("text", ""))
        start, n = 0, len(cleaned)
        while start < n:
            end = min(start + chunk_size, n)
            # Ajustement léger vers une frontière de phrase, sans bloquer la progression.
            if end < n:
                zone_start = max(start, end - 80)
                matches = list(re.finditer(r'[.!?]\\s', cleaned[zone_start:end]))
                if matches:
                    end = zone_start + matches[-1].end()
            if end <= start:
                end = min(start + chunk_size, n)
            chunk_text = cleaned[start:end].strip()
            if chunk_text:
                chunks.append({"chunk_id": chunk_id, "text": chunk_text,
                    "metadata": dict(doc.get("metadata", {}))})
                chunk_id += 1
            if end >= n:
                break
            next_start = end - chunk_overlap
            start = max(start + 1, next_start)
    return chunks

# 8. Génération des embeddings & 9. Création de l'index FAISS

Nous chargeons le modèle sémantique performant et léger `all-MiniLM-L6-v2` pour projeter nos chunks de texte dans un espace vectoriel à 384 dimensions, puis nous les indexons à l'aide de **FAISS** (Flat L2 index) pour des recherches instantanées.

In [45]:
# Chargement du modèle d'embeddings
print("Chargement du modèle d'embeddings...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
EMBEDDING_DIM = int(embedder.get_sentence_embedding_dimension())

class RAGIndex:
    def __init__(self, dimension=EMBEDDING_DIM):
        self.dimension = dimension
        self.index = faiss.IndexFlatL2(dimension)
        self.chunks_store = []
        # Identifiants basés sur hash du contenu du fichier, pas seulement le nom.
        self.indexed_files = set()

    def add_chunks(self, chunks, file_key=None):
        if not chunks or (file_key and file_key in self.indexed_files):
            return 0
        texts = [c["text"] for c in chunks if c.get("text", "").strip()]
        chunks = [c for c in chunks if c.get("text", "").strip()]
        if not chunks: return 0
        embeddings = np.asarray(embedder.encode(texts, convert_to_numpy=True), dtype=np.float32)
        if embeddings.ndim != 2 or embeddings.shape[1] != self.dimension:
            raise ValueError(f"Dimension embeddings {embeddings.shape} incompatible avec {self.dimension}.")
        faiss.normalize_L2(embeddings)
        # Ajout FAISS et stockage dans le même ordre, seulement après validation.
        self.index.add(embeddings)
        self.chunks_store.extend(chunks)
        if file_key: self.indexed_files.add(file_key)
        assert self.index.ntotal == len(self.chunks_store)
        return len(chunks)

rag_index = RAGIndex()

Chargement du modèle d'embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

# 10. Recherche sémantique

Cette fonction convertit une requête utilisateur en vecteur, effectue une recherche de proximité dans l'index FAISS et extrait les $K$ chunks les plus pertinents avec leurs métadonnées.

In [46]:
def search_index(query, k=3):
    if not isinstance(query, str) or not query.strip() or not rag_index.chunks_store:
        return []
    try: k = int(k)
    except (TypeError, ValueError): return []
    if k <= 0: return []
    k = min(k, len(rag_index.chunks_store))
    qvec = np.asarray(embedder.encode([query.strip()], convert_to_numpy=True), dtype=np.float32)
    if qvec.shape[1] != rag_index.dimension: raise ValueError("Dimension de requête incompatible avec FAISS.")
    faiss.normalize_L2(qvec)
    distances, indices = rag_index.index.search(qvec, k)
    return [{"chunk": rag_index.chunks_store[idx], "distance": float(score)}
            for score, idx in zip(distances[0], indices[0])
            if idx >= 0 and idx < len(rag_index.chunks_store)]

# 11. Génération des réponses avec Gemini & 12. Pipeline RAG complet

Le pipeline complet prend la question de l'utilisateur, interroge l'index FAISS, assemble le prompt avec un cadre strict d'interdiction d'hallucination, puis appelle l'API Gemini (modèle `modèle vérifié lors de la configuration`).

In [47]:
def generate_answer_with_rag(query, k=3):
    if not query or not query.strip():
        return "Veuillez saisir une question.", []
    if client is None or available_gemini_model is None:
        return "Gemini n'est pas vérifié/configuré. Exécutez la cellule de configuration avec une clé API valide.", []
    results = search_index(query, k=k)
    if not results:
        return "Aucun passage récupéré. Indexez des PDF et vérifiez votre question.", []
    context_parts, sources = [], []
    for i, result in enumerate(results, 1):
        chunk = result["chunk"]
        source = f"{chunk['metadata'].get('source','Inconnu')} (page {chunk['metadata'].get('page','?')})"
        context_parts.append(f"[Passage {i} | Source: {source}]\\n{chunk['text']}")
        sources.append(source)
    prompt = ("Réponds en français uniquement à partir du contexte. Ne complète pas avec des connaissances externes. "
              "Si le contexte ne contient pas la réponse, dis-le explicitement.\\n\\n"
              + "\\n\\n".join(context_parts) + f"\\n\\nQuestion : {query}")
    try:
        response = client.models.generate_content(model=available_gemini_model, contents=prompt)
        return (response.text or "Réponse vide du modèle."), list(dict.fromkeys(sources))
    except Exception as e:
        code = getattr(e, "status_code", None)
        if code == 404: msg = "Erreur 404 : modèle ou endpoint indisponible."
        elif code == 429: msg = "Erreur 429 : quota ou limite de requêtes dépassée."
        elif code == 503: msg = "Erreur 503 : service temporairement indisponible."
        else: msg = f"Erreur API ({code or type(e).__name__}) : {e}"
        return msg, list(dict.fromkeys(sources))

# 13. Interface utilisateur (Gradio)

Nous mettons en place une interface Gradio intuitive et moderne directement intégrable dans Google Colab pour charger les documents et dialoguer avec le système.

In [48]:
def _file_path_and_name(file_obj):
    # Gradio File peut renvoyer un filepath, un objet ayant .name, ou une dict.
    if isinstance(file_obj, dict):
        path = file_obj.get("path") or file_obj.get("name")
        display_name = file_obj.get("orig_name") or file_obj.get("name") or os.path.basename(str(path))
    else:
        path = getattr(file_obj, "path", None) or getattr(file_obj, "name", None) or str(file_obj)
        display_name = getattr(file_obj, "orig_name", None) or os.path.basename(str(path))
    if not path: raise ValueError("Chemin de fichier absent.")
    return str(path), str(display_name)

def process_pdf_and_index(file_objs):
    if not file_objs: return "Aucun PDF fourni."
    files = file_objs if isinstance(file_objs, list) else [file_objs]
    added_files, added_chunks, skipped = 0, 0, 0
    for file_obj in files:
        try:
            filepath, filename = _file_path_and_name(file_obj)
            with open(filepath, "rb") as f: digest = hashlib.sha256(f.read()).hexdigest()
            file_key = digest
            if file_key in rag_index.indexed_files:
                skipped += 1; continue
            extracted = extract_text_from_pdf(filepath, source_name=filename)
            chunks = create_chunks(extracted, start_id=len(rag_index.chunks_store))
            if chunks:
                count = rag_index.add_chunks(chunks, file_key=file_key)
                added_chunks += count
                added_files += bool(count)
        except Exception as e:
            print(f"Erreur traitement fichier : {type(e).__name__}: {e}")
    if not added_chunks:
        return f"Aucun nouveau chunk indexé. Fichiers déjà traités : {skipped}."
    return f"Indexation terminée : {added_files} fichier(s), {added_chunks} chunks. Total : {len(rag_index.chunks_store)}."

def gradio_ask(query):
    answer, sources = generate_answer_with_rag(query)
    return answer, "\\n".join(f"- {s}" for s in sources) if sources else "Aucune source."

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🧠 RAG Intelligent PDF Assistant")
    gr.Markdown("Chargez un ou plusieurs PDF puis posez vos questions sur leur contenu.")
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Étape 1 — Documents")
            file_input = gr.File(label="PDF", file_count="multiple", file_types=[".pdf"], type="filepath")
            upload_btn = gr.Button("Indexer les documents", variant="primary")
            status_output = gr.Textbox(label="Statut", interactive=False)
        with gr.Column(scale=2):
            gr.Markdown("### Étape 2 — Question")
            query_input = gr.Textbox(label="Question", placeholder="Posez votre question...")
            ask_btn = gr.Button("Rechercher et répondre")
            answer_output = gr.Textbox(label="Réponse", lines=8, interactive=False)
            sources_output = gr.Textbox(label="Sources (fichier et page)", lines=4, interactive=False)
    upload_btn.click(process_pdf_and_index, inputs=file_input, outputs=status_output)
    ask_btn.click(gradio_ask, inputs=query_input, outputs=[answer_output, sources_output])

print("Interface définie. Lancez demo.launch(inline=True, share=False) dans Colab.")
demo.launch(inline=True, share=False)

Interface Gradio 6 prête pour lancement via 'demo.launch(theme=gr.themes.Soft())'.


# 14. Évaluation du système

L'évaluation d'un système RAG passe principalement par la mesure de la qualité de son système de récupération (Retriever). Les trois métriques standards et académiques sont :

1. **Precision@K** : Proportion de documents récupérés qui sont effectivement pertinents par rapport à la question.
2. **Recall@K** : Proportion des documents pertinents totaux qui ont été effectivement récupérés par l'index.
3. **Mean Reciprocal Rank (MRR)** : Évalue la position (le rang) du premier document pertinent récupéré. Plus le document le plus pertinent est haut dans la liste des résultats, plus le score est proche de 1.

Nous implémentons ci-dessous un script d'évaluation prêt à l'emploi.

In [49]:
def evaluate_retrieval(evaluation_dataset, k=3):
    if not rag_index.chunks_store:
        return "Index vide : indexez des PDF avant l'évaluation."
    if not evaluation_dataset:
        return "Aucun jeu de test valide fourni. Complétez mon_jeu_de_test avec des questions et les IDs réels des chunks pertinents."
    if not isinstance(k, int) or k <= 0:
        return "K doit être un entier positif."
    precisions, recalls, reciprocal_ranks = [], [], []
    valid_items = 0
    for item in evaluation_dataset:
        query = item.get("query", "")
        expected = item.get("expected_chunk_ids", [])
        if not query.strip() or not expected or not all(isinstance(x, int) for x in expected):
            continue
        expected_ids = set(expected)
        results = search_index(query, k=k)
        retrieved_ids = [r["chunk"]["chunk_id"] for r in results]
        hits = sum(rid in expected_ids for rid in retrieved_ids)
        precisions.append(hits / len(retrieved_ids) if retrieved_ids else 0.0)
        recalls.append(hits / len(expected_ids) if expected_ids else 0.0)
        reciprocal_ranks.append(next((1.0/(rank+1) for rank, rid in enumerate(retrieved_ids) if rid in expected_ids), 0.0))
        valid_items += 1
    if not valid_items:
        return "Aucun exemple valide : indiquez query et expected_chunk_ids réels."
    return {"Precision@K": float(np.mean(precisions)), "Recall@K": float(np.mean(recalls)), "MRR": float(np.mean(reciprocal_ranks))}

# À compléter uniquement avec des IDs de chunks constatés dans l'index, jamais supposés.
mon_jeu_de_test = []
print(evaluate_retrieval(mon_jeu_de_test, k=3))

L'index est actuellement vide. Veuillez indexer un document PDF via l'interface ci-dessus avant de lancer l'évaluation.


# 15. Conclusion et limites

### Conclusion
Ce projet présente la création d'un pipeline RAG complet et robuste. L'utilisation combinée d'un extracteur fiable (`PyMuPDF`), d'un index vectoriel ultra-performant (`FAISS`) et d'un LLM de pointe via l'API officielle (`modèle Gemini vérifié via l'API`) permet de construire un assistant académique hautement qualifié.

### Limites techniques identifiées :
1. **Sensibilité au bruit** : Le découpage par caractères fixes (caractères / tokens) peut parfois couper des phrases importantes en plein milieu.
2. **Dépendance au modèle de plongement (Embeddings)** : Les performances du retriever dépendent fortement de la capacité du modèle `all-MiniLM-L6-v2` à comprendre le vocabulaire très spécifique d'un document métier.
3. **Gestion des tableaux et graphiques** : PyMuPDF extrait principalement le texte brut ; les tableaux et schémas complexes peuvent requérir des modèles de vision (Multimodal RAG).

## Fichiers annexes

Les fichiers `README.md` et `requirements.txt` sont également fournis à côté du notebook dans le dossier du projet.